# Data Description
- Background: 警察局交通大隊提供之死傷事故明細。新研究涵蓋兩年全天，以行政里事故組成與環境比較為主；此處完整記錄所有原始欄位。
- date received: 未知；本機下載／收到日期待學生或原始下載紀錄補證，不以文件建立日代填。
- Path to data file: [臺北市死傷交通事故資料官方資料集](https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735)；來源網址由學生提供，2026-11-25 唯讀查閱詮釋資料，未重新下載原始資料。PDF 下載來源與本機位元組關係未認證。
- Unit of observation: 2024 每列當事人紀錄；2025 本機每列均為第一當事人。候選事件採第一當事人，無官方事故 ID。
- Sample period: 民國 113／114 年，日期與月份涵蓋依本次輸出核對；非 PDF 自身的樣本期間。
- Known issues: 跨年欄位及當事人口徑不同；3 筆候選紀錄缺座標；CRS 未認證；傷亡缺值、速限及年齡特殊值待驗證。45,130 是候選紀錄，不是完成歸里的樣本。
- Definition for each variable: 見 Summary Stats 的完整逐欄定義表；「觀測內容」與「官方定義」分列，官方未定義處明記未知。
- Data path: [113年-臺北市死傷交通事故明細.csv](113年-臺北市死傷交通事故明細.csv)、[114年-臺北市死傷交通事故明細.csv](114年-臺北市死傷交通事故明細.csv)、[事故對照表.pdf](事故對照表.pdf)；程式讀檔使用下方精確的工作包相對路徑。
- Role: `raw`（參考 PDF 亦逐檔登記）
- Verification status: 已實際執行唯讀文件化檢查；詳見保存輸出。檔案身分、基本結構及列明診斷可核對，不認證為 analysis-ready。
- SHA-256: 見下方固定綁定值及 Summary 逐檔大小／hash 表；每次讀檔前後重新計算。
- Next processing notebook: pending；尚未建立，不預設處理檔名。
- Author / contributors: 詹明翰、謝子詮、曾翊涵（學生確認三人共同完成本次 Unit 0.5；不推定個別分工）
- Created At: 2026-11-25（文件建立日，與 date received 不同）

**官方版本資訊與本機版本分開記錄：** 官方頁提供機關為警察局交通大隊；113／114 資源更新時間分別為 2026-06-11 14:44:01／14:44:07，詮釋資料更新 2026-06-11 14:45:22。此為查閱當時網頁資訊，不是本機下載日期或位元組版本證明。PDF 本機印有 112.07.01 後適用；2024–2025 雖在日期後，發布 CSV 映射仍待確認。

本文件依 `templates/data/raw/template-doc-and-summary-stats.ipynb` 的原欄位與章節順序建立。
僅描述未修改原始資料；未刪除、補值、修復、正式轉換或輸出 processed。正式研究與規劃定義詳見 [PROJECT.md](../../PROJECT.md)。


In [1]:
from pathlib import Path
import hashlib, json, sys, platform, os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('Open within the project containing AGENTS.md and templates/.')

PROJECT_ROOT = find_project_root()
assert Path(sys.executable).resolve() == (PROJECT_ROOT / '.venv/Scripts/python.exe').resolve()
assert sys.prefix != sys.base_prefix
TEMP = PROJECT_ROOT / 'data/temp/unit05_documentation'
for variable, folder in {'MPLCONFIGDIR':'matplotlib', 'IPYTHONDIR':'ipython',
                         'JUPYTER_RUNTIME_DIR':'runtime'}.items():
    location = TEMP / folder
    location.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(location)
os.environ['PROJ_NETWORK'] = 'OFF'
pd.set_option('display.max_colwidth', 100)

def project_file(relative_path, area='raw'):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('Explicit project-relative path required')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('Input path outside permitted data area')
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify_inputs(expected):
    rows = []
    for relative_path, digest in expected.items():
        path = project_file(relative_path)
        actual = sha256_file(path)
        if actual != digest:
            raise ValueError(f'Input changed; review provenance before loading: {relative_path}')
        rows.append({'Data path':relative_path, 'bytes':path.stat().st_size,
                     'format':path.suffix.lower(), 'SHA-256':actual})
    return pd.DataFrame(rows)

def show_table(title, frame, block=25):
    display(Markdown('**' + title + '**'))
    if not len(frame):
        display(Markdown('無符合紀錄。'))
    else:
        for start in range(0, len(frame), block):
            display(frame.iloc[start:start + block])

def sample_ten(title, frame, block_columns=12):
    display(Markdown('**' + title + '：實際前十列，按欄分組呈現；未抽樣或排序**'))
    for start in range(0, len(frame.columns), block_columns):
        display(frame.iloc[:10, start:start + block_columns])

def schema(frame, definitions, levels=None):
    rows=[]
    for column in frame.columns:
        meaning, evidence = definitions[column]
        rows.append({'欄位':column, '讀入型態':str(frame[column].dtype),
                     '定義或可觀察內容':meaning, '定義依據及限制':evidence,
                     '層級':levels.get(column, '原始紀錄') if levels else '原始紀錄',
                     '缺失數':int(frame[column].isna().sum()),
                     '非缺失相異值':int(frame[column].nunique(dropna=True))})
    return pd.DataFrame(rows)

def finish_check():
    for relative_path,digest in EXPECTED_SHA256.items():
        assert sha256_file(project_file(relative_path)) == digest, relative_path
    print({'input_hashes_unchanged':True, 'input_files_checked':len(EXPECTED_SHA256),
           'interpreter_matches_project_venv':True, 'python_version':platform.python_version(),
           'scope':'read-only raw documentation; no processed dataset or model output'})


## Change for Your Own Data
以下固定原始檔路徑與事前檢查所得 SHA-256；版本不同即停止，不換用其他檔案。
CSV 的編碼／代碼型態、Shapefile 的 reader 與配套檔均明列。讀入型態是本次 reader 的推斷或明示設定，不冒稱官方儲存型態。


In [2]:
EXPECTED_SHA256 = {'data/raw/113年-臺北市死傷交通事故明細.csv': 'b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5', 'data/raw/114年-臺北市死傷交通事故明細.csv': '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c', 'data/raw/事故對照表.pdf': 'f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670'}
INPUT_MANIFEST = verify_inputs(EXPECTED_SHA256)


## Summary


In [3]:
show_table('表 1：原始輸入逐檔檔案格式、大小及 SHA-256', INPUT_MANIFEST)
frames={year:pd.read_csv(project_file(path),encoding='utf-8-sig',low_memory=False)
         for year,path in {2024:'data/raw/113年-臺北市死傷交通事故明細.csv',2025:'data/raw/114年-臺北市死傷交通事故明細.csv'}.items()}
show_table('表 2：事故原始結構',pd.DataFrame([{'年度':year,'列數':len(f),'欄數':len(f.columns),'編碼':'UTF-8-SIG'} for year,f in frames.items()]))
shared=[c for c in frames[2024] if c in frames[2025]]
print({'共同欄位數':len(shared),'共同欄位':shared,
       '僅2024':[c for c in frames[2024] if c not in frames[2025]],
       '僅2025':[c for c in frames[2025] if c not in frames[2024]]})


**表 1：原始輸入逐檔檔案格式、大小及 SHA-256**

,Data path,bytes,format,SHA-256
0,data/raw/113年-臺北市死傷交通事故明細.csv,9398043,.csv,b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5
1,data/raw/114年-臺北市死傷交通事故明細.csv,4082763,.csv,61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c
2,data/raw/事故對照表.pdf,487979,.pdf,f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670


**表 2：事故原始結構**

,年度,列數,欄數,編碼
0,2024,51810,49,UTF-8-SIG
1,2025,22762,47,UTF-8-SIG


{'共同欄位數': 46, '共同欄位': ['發生年度', '發生月', '發生日', '發生時-Hours', '發生分', '區序', '肇事地點', '死亡人數', '2-30日死亡人數', '受傷人數', '當事人序號', '車種', '天候', '道路類別', '速限-速度限制', '道路型態', '事故位置', '路面狀況1', '路面狀況2', '路面狀況3', '道路障礙1', '道路障礙2', '號誌1', '號誌2', '車道劃分-分向', '車道劃分-分道1', '車道劃分-分道2', '車道劃分-分道3', '事故類型及型態', '性別', '年齡', '受傷程度', '主要傷處', '保護裝置', '行動電話', '車輛用途', '當事者行動狀態', '駕駛資格情形', '駕駛執照種類', '飲酒情形', '車輛撞擊部位1', '肇因碼-個別', '個人肇逃否', '職業', '座標-X', '座標-Y'], '僅2024': ['道路照明設備', '車輛撞擊部位2', '肇因碼-主要'], '僅2025': ['光線']}


### Sample


In [4]:
for year,frame in frames.items():
    sample_ten(f'表 3：{year} 原始樣本',frame)


**表 3：2024 原始樣本：實際前十列，按欄分組呈現；未抽樣或排序**

,發生年度,發生月,發生日,發生時-Hours,發生分,區序,肇事地點,死亡人數,2-30日死亡人數,受傷人數,當事人序號,車種
0,113,1,1,0,22,07信義區,信義區基隆路1段147巷5弄11號,0.0,0.0,1.0,1,C03
1,113,1,1,0,22,07信義區,信義區基隆路1段147巷5弄11號,0.0,0.0,1.0,2,H01
2,113,1,1,0,36,05中正區,中正區信義路2段與信義路2段181巷口,0.0,0.0,1.0,1,B03
3,113,1,1,0,36,05中正區,中正區信義路2段與信義路2段181巷口,0.0,0.0,1.0,2,C02
4,113,1,1,0,45,06松山區,松山區八德路4段389巷與南京東路5段250巷18弄口,0.0,0.0,1.0,2,H01
5,113,1,1,0,45,06松山區,松山區八德路4段389巷與南京東路5段250巷18弄口,0.0,0.0,1.0,1,B12
6,113,1,1,2,10,06松山區,松山區光復南路71號,0.0,0.0,1.0,1,C04
7,113,1,1,2,10,06松山區,松山區光復南路71號,0.0,0.0,1.0,2,H01
8,113,1,1,2,27,04大安區,大安區復興南路2段362號,0.0,0.0,1.0,1,H01
9,113,1,1,2,27,04大安區,大安區復興南路2段362號,0.0,0.0,1.0,2,B03


,天候,道路照明設備,道路類別,速限-速度限制,道路型態,事故位置,路面狀況1,路面狀況2,路面狀況3,道路障礙1,道路障礙2,號誌1
0,8,5.0,5.0,30.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
1,8,5.0,5.0,30.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
2,8,5.0,5.0,50.0,3.0,1.0,1.0,5.0,4.0,5.0,13.0,4.0
3,8,5.0,5.0,50.0,3.0,1.0,1.0,5.0,4.0,5.0,13.0,4.0
4,7,5.0,5.0,30.0,3.0,2.0,1.0,5.0,4.0,5.0,13.0,4.0
5,7,5.0,5.0,30.0,3.0,2.0,1.0,5.0,4.0,5.0,13.0,4.0
6,7,5.0,5.0,50.0,4.0,18.0,1.0,5.0,4.0,5.0,13.0,2.0
7,7,5.0,5.0,50.0,4.0,18.0,1.0,5.0,4.0,5.0,13.0,2.0
8,8,5.0,5.0,50.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
9,8,5.0,5.0,50.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0


,號誌2,車道劃分-分向,車道劃分-分道1,車道劃分-分道2,車道劃分-分道3,事故類型及型態,性別,年齡,受傷程度,主要傷處,保護裝置,行動電話
0,4.0,8.0,3.0,5.0,2.0,2.0,1,22.0,3.0,10.0,3.0,4.0
1,4.0,8.0,3.0,5.0,2.0,2.0,2,47.0,2.0,9.0,4.0,5.0
2,4.0,1.0,4.0,5.0,2.0,16.0,1,52.0,3.0,10.0,7.0,1.0
3,4.0,1.0,4.0,5.0,2.0,16.0,1,47.0,2.0,9.0,6.0,1.0
4,4.0,10.0,5.0,5.0,2.0,9.0,1,56.0,2.0,7.0,4.0,5.0
5,4.0,10.0,5.0,5.0,2.0,9.0,2,41.0,3.0,10.0,7.0,1.0
6,1.0,1.0,3.0,4.0,1.0,1.0,1,23.0,3.0,10.0,5.0,1.0
7,1.0,1.0,3.0,4.0,1.0,1.0,1,52.0,2.0,9.0,4.0,5.0
8,4.0,1.0,5.0,5.0,1.0,3.0,1,22.0,2.0,9.0,4.0,5.0
9,4.0,1.0,5.0,5.0,1.0,3.0,1,43.0,3.0,10.0,7.0,1.0


,車輛用途,當事者行動狀態,駕駛資格情形,駕駛執照種類,飲酒情形,車輛撞擊部位1,車輛撞擊部位2,肇因碼-個別,肇因碼-主要,個人肇逃否,職業,座標-X
0,7.0,9.0,1.0,10.0,1.0,11.0,NaN,4.0,NaN,1.0,21.0,121.566408
1,8.0,16.0,8.0,20.0,10.0,16.0,NaN,60.0,NaN,1.0,22.0,121.566408
2,7.0,5.0,1.0,8.0,2.0,8.0,NaN,80.0,NaN,1.0,21.0,121.529308
3,7.0,9.0,1.0,9.0,2.0,12.0,NaN,44.0,NaN,1.0,21.0,121.529308
4,8.0,16.0,8.0,20.0,10.0,16.0,NaN,67.0,NaN,1.0,21.0,121.568319
5,7.0,5.0,1.0,8.0,2.0,2.0,NaN,108.0,NaN,1.0,21.0,121.568319
6,7.0,9.0,1.0,10.0,1.0,11.0,NaN,70.0,NaN,1.0,21.0,121.557867
7,8.0,18.0,8.0,20.0,1.0,16.0,NaN,121.0,NaN,1.0,21.0,121.557867
8,8.0,16.0,8.0,20.0,10.0,16.0,NaN,121.0,NaN,1.0,21.0,121.543339
9,7.0,9.0,1.0,8.0,1.0,4.0,NaN,44.0,NaN,1.0,21.0,121.543339


,座標-Y
0,25.043090
1,25.043090
2,25.033923
3,25.033923
4,25.050199
5,25.050199
6,25.045344
7,25.045344
8,25.021828
9,25.021828


**表 3：2025 原始樣本：實際前十列，按欄分組呈現；未抽樣或排序**

,發生年度,發生月,發生日,發生時-Hours,發生分,區序,肇事地點,死亡人數,2-30日死亡人數,受傷人數,當事人序號,車種
0,114,1,1,0,18,03中山區,中山區市民大道2段47號,0.0,0.0,1.0,1,C03
1,114,1,1,0,31,10文山區,文山區新光路1段109號旁,0.0,0.0,3.0,1,C03
2,114,1,1,0,35,03中山區,中山區北安路與大直街口,0.0,0.0,1.0,1,C03
3,114,1,1,0,41,03中山區,中山區民族東路2號,0.0,0.0,1.0,1,B01
4,114,1,1,0,57,11南港區,南港區忠孝東路6段與玉成街口,0.0,0.0,1.0,1,B03
5,114,1,1,1,34,07信義區,信義區吳興街269巷18弄與吳興街281巷47弄口,0.0,0.0,3.0,1,C04
6,114,1,1,2,15,07信義區,信義區基隆路2段107號之2,0.0,0.0,1.0,1,C03
7,114,1,1,6,26,10文山區,文山區指南路3段(岐山幹42),0.0,0.0,1.0,1,C03
8,114,1,1,6,29,10文山區,文山區辛亥路4段與辛亥路4段101巷口,0.0,0.0,1.0,1,B01
9,114,1,1,7,11,06松山區,松山區寶清街93號,0.0,0.0,2.0,1,B02


,天候,光線,道路類別,速限-速度限制,道路型態,事故位置,路面狀況1,路面狀況2,路面狀況3,道路障礙1,道路障礙2,號誌1
0,8.0,5.0,5.0,40.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
1,7.0,5.0,5.0,50.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
2,7.0,5.0,5.0,50.0,5.0,1.0,1.0,5.0,4.0,5.0,13.0,2.0
3,7.0,5.0,5.0,50.0,14.0,7.0,1.0,5.0,4.0,5.0,13.0,4.0
4,7.0,5.0,5.0,50.0,3.0,1.0,1.0,5.0,4.0,5.0,12.0,2.0
5,7.0,5.0,5.0,30.0,3.0,1.0,1.0,5.0,4.0,5.0,13.0,4.0
6,7.0,5.0,5.0,50.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
7,8.0,7.0,5.0,30.0,12.0,9.0,1.0,5.0,4.0,5.0,13.0,4.0
8,8.0,6.0,5.0,50.0,3.0,2.0,1.0,5.0,4.0,5.0,13.0,2.0
9,8.0,5.0,5.0,30.0,14.0,9.0,1.0,5.0,4.0,5.0,13.0,1.0


,號誌2,車道劃分-分向,車道劃分-分道1,車道劃分-分道2,車道劃分-分道3,事故類型及型態,性別,年齡,受傷程度,主要傷處,保護裝置,行動電話
0,4.0,1.0,4.0,5.0,2.0,13.0,1.0,22.0,2.0,9.0,5.0,1.0
1,4.0,9.0,5.0,5.0,2.0,17.0,1.0,21.0,2.0,8.0,6.0,1.0
2,1.0,1.0,5.0,5.0,1.0,16.0,2.0,21.0,2.0,9.0,3.0,4.0
3,4.0,6.0,3.0,4.0,1.0,17.0,1.0,54.0,3.0,10.0,7.0,1.0
4,1.0,3.0,4.0,5.0,2.0,13.0,1.0,19.0,3.0,10.0,3.0,4.0
5,4.0,10.0,4.0,5.0,1.0,15.0,2.0,25.0,2.0,8.0,3.0,4.0
6,4.0,1.0,4.0,5.0,2.0,13.0,1.0,18.0,2.0,9.0,6.0,1.0
7,4.0,10.0,5.0,5.0,1.0,10.0,1.0,21.0,2.0,1.0,6.0,1.0
8,1.0,1.0,4.0,5.0,2.0,12.0,1.0,66.0,3.0,10.0,7.0,1.0
9,1.0,9.0,4.0,5.0,1.0,12.0,1.0,44.0,3.0,10.0,7.0,1.0


,車輛用途,當事者行動狀態,駕駛資格情形,駕駛執照種類,飲酒情形,車輛撞擊部位1,肇因碼-個別,個人肇逃否,職業,座標-X,座標-Y
0,7.0,9.0,1.0,10.0,1.0,11.0,16.0,1.0,21.0,121.529004,25.046603
1,7.0,1.0,1.0,10.0,1.0,14.0,19.0,1.0,21.0,121.573689,24.991275
2,7.0,5.0,3.0,18.0,1.0,12.0,96.0,1.0,21.0,121.547196,25.079842
3,7.0,1.0,1.0,4.0,1.0,8.0,19.0,1.0,21.0,121.523305,25.068395
4,7.0,9.0,1.0,8.0,1.0,5.0,108.0,1.0,22.0,121.583227,25.045872
5,7.0,15.0,1.0,10.0,1.0,11.0,74.0,1.0,21.0,121.565200,25.026933
6,7.0,9.0,1.0,10.0,1.0,11.0,118.0,1.0,22.0,121.557572,25.029541
7,7.0,9.0,1.0,10.0,1.0,11.0,108.0,1.0,21.0,121.597223,24.972428
8,7.0,7.0,1.0,4.0,2.0,2.0,100.0,1.0,21.0,121.558397,25.006447
9,7.0,1.0,1.0,8.0,1.0,4.0,19.0,1.0,21.0,121.567989,25.052453


### Summary Stats


In [5]:
DEFINITIONS = {'發生年度': ('事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911', 'PDF p.1 發生時間；原始值與檔名核對'), '發生月': ('事故發生月份', 'PDF p.1 發生時間'), '發生日': ('事故發生日', 'PDF p.1 發生時間'), '發生時-Hours': ('事故發生小時；檔內 24 小時制數值', 'PDF p.1 發生時間；檔內有效性檢查'), '發生分': ('事故發生分鐘', 'PDF p.1 發生時間'), '區序': ('本機字串含行政區序號及區名；正式區序代碼規格未知', '官方頁列欄名；檔內觀測，不自行當里代碼'), '肇事地點': ('事故地點文字；不是當事人居住地址', 'PDF p.1 發生地點；發布 CSV 轉錄細節未知'), '死亡人數': ('事故死傷人數中的 24 小時內死亡人數；不含個人受傷程度代碼', 'PDF p.1 死傷人數；與官方頁 24 小時定義對照'), '2-30日死亡人數': ('欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認', 'CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式'), '受傷人數': ('事故受傷人數；與單一當事人的受傷程度不同', 'PDF p.1 死傷人數'), '當事人序號': ('事故內當事人序次的候選欄；1 為第一當事人紀錄', 'PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定'), '車種': ('當事者區分類別，包括車輛及人', 'PDF p.3 類別表；如 C03 普通重型機車、H01 行人'), '天候': ('事故天候代碼', 'PDF p.1 天候；例 6 雨、7 陰、8 晴'), '道路照明設備': ('2024 欄名；與 PDF 照明欄候選對應', 'PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認'), '光線': ('2025 欄名；與道路照明設備候選對應', 'PDF p.1 道路照明設備；CSV 名稱不同，不能直接視為已認證跨年等價'), '道路類別': ('事故道路類別代碼，表單註明第 1 當事者', 'PDF p.1；例 5 市區道路'), '速限-速度限制': ('道路速限，不是當事人實際車速；0、502 等值不自行更正', 'PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認'), '道路型態': ('事故道路型態代碼', 'PDF p.1；例 03 三岔、04 四岔、14 直路'), '事故位置': ('事故發生位置類別', 'PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區'), '路面狀況1': ('路面鋪裝代碼', 'PDF p.1 路面狀況 (1)'), '路面狀況2': ('路面狀態代碼', 'PDF p.1 路面狀況 (2)；5 乾燥'), '路面狀況3': ('路面缺陷代碼', 'PDF p.1 路面狀況 (3)；4 無缺陷'), '道路障礙1': ('障礙物代碼', 'PDF p.1 道路障礙 (1)；5 無障礙物'), '道路障礙2': ('視距代碼；新版及表單內舊碼不可混用', 'PDF p.1 道路障礙 (2) 同時出現版本相關項目；13 良好'), '號誌1': ('號誌種類代碼', 'PDF p.1 號誌 (1)；4 無號誌'), '號誌2': ('號誌動作代碼', 'PDF p.1 號誌 (2)；1 正常、4 無號誌'), '車道劃分-分向': ('分向設施代碼', 'PDF p.1 分向設施；10 無分向設施'), '車道劃分-分道1': ('快車道或一般車道間分道設施代碼', 'PDF p.1 分道設施 (1)'), '車道劃分-分道2': ('快慢車道間分道設施代碼', 'PDF p.1 分道設施 (2)'), '車道劃分-分道3': ('路面邊線代碼', 'PDF p.1 分道設施 (3)；1 有、2 無'), '事故類型及型態': ('事故情境分類，不等同所有涉入者的完整屬性', 'PDF pp.1–2；1–9 人與車、13 追撞、16 側撞'), '性別': ('當事者屬性別代碼，含非自然人及未查獲等特殊類別', 'PDF p.3；1 男、2 女、3 無或物、4 肇逃未查獲、5 無人駕駛'), '年齡': ('當事者年齡數值；年齡基準、-1／0 等特殊值定義未知', 'PDF p.3 原表為出生年月日；CSV 年齡轉換及特殊值需補證'), '受傷程度': ('單一當事者傷亡程度，不是事故傷亡總數，也無法直接辨識重傷', 'PDF p.3；1 24小時內死亡、2 受傷、3 未受傷、4 不明、5 2–30日內死亡'), '主要傷處': ('當事者主要受傷部位代碼', 'PDF p.3；09 多數傷、10 無、11 不明'), '保護裝置': ('當事者保護裝備使用類別', 'PDF p.3 保護裝備；3 不明、4 無需使用等，版本映射仍需驗證'), '行動電話': ('行動電話、電腦或類似裝置使用情形', 'PDF p.3；1 未使用、4 不明、5 非駕駛人'), '車輛用途': ('車輛用途／非駕駛人類別', 'PDF p.3；7 其他、8 非駕駛人及乘客'), '當事者行動狀態': ('車或人的事故當時行動狀態', 'PDF p.3；9 向前直行、16 步行、21 不明'), '駕駛資格情形': ('駕照狀態代碼', 'PDF p.3；1 正常駕照、7 不明、8 無需駕照之人'), '駕駛執照種類': ('駕照種類代碼', 'PDF p.3；18 無駕照、19 不明、20 無需駕照之人'), '飲酒情形': ('酒精觀察／檢測分類，不是連續酒測值', 'PDF p.3；1 觀察未飲酒、2 無反應、9 無法檢測、10 非駕駛人、11 不明'), '車輛撞擊部位1': ('最初撞擊部位候選對應', 'PDF p.3 車輛撞擊部位 最初／其他；CSV 對應待發布規格確認'), '車輛撞擊部位2': ('其他撞擊部位候選對應；僅 2024 欄', 'PDF p.3；與欄位 1 的發布規則待確認'), '肇因碼-個別': ('初步分析研判之個別參照碼，不是法院責任或因果效應', 'PDF pp.3–4 個別初步分析研判索引'), '肇因碼-主要': ('主要初步分析研判候選欄；僅 2024 且全缺失', 'PDF pp.3–4；發布填值及跨年移除規則未知'), '個人肇逃否': ('當事者肇事逃逸類別', 'PDF p.3；1 否、2 是'), '職業': ('當事者職業代碼；本機碼與 PDF 不完整對應', 'PDF p.3 僅有部分項目；完整 CSV 職業碼表未知'), '座標-X': ('原始點位 X 數值；範圍近似經度但座標基準未確認', '官方頁僅稱 X、Y；不據範圍自行指定 EPSG'), '座標-Y': ('原始點位 Y 數值；範圍近似緯度但座標基準未確認', '官方頁僅稱 X、Y；不據範圍自行指定 EPSG')}
event_columns = list(frames[2024].columns[:30])
event_columns = [c for c in event_columns if c not in ['當事人序號','車種']]
event_columns += ['座標-X','座標-Y','肇因碼-主要','光線']
for year,frame in frames.items():
    levels={c:('事故情境／事故總量（候選）' if c in event_columns else '當事人（候選）') for c in frame}
    show_table(f'表 4：{year} 完整欄位、型態、定義、缺失及層級',schema(frame,DEFINITIONS,levels),block=16)
    show_table(f'表 5：{year} 原始描述統計（代碼均值不作實質解讀）',frame.describe(include='all').T,block=20)


**表 4：2024 完整欄位、型態、定義、缺失及層級**

,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
0,發生年度,int64,事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911,PDF p.1 發生時間；原始值與檔名核對,事故情境／事故總量（候選）,0,1
1,發生月,int64,事故發生月份,PDF p.1 發生時間,事故情境／事故總量（候選）,0,12
2,發生日,int64,事故發生日,PDF p.1 發生時間,事故情境／事故總量（候選）,0,31
3,發生時-Hours,int64,事故發生小時；檔內 24 小時制數值,PDF p.1 發生時間；檔內有效性檢查,事故情境／事故總量（候選）,0,24
4,發生分,int64,事故發生分鐘,PDF p.1 發生時間,事故情境／事故總量（候選）,0,60
5,區序,str,本機字串含行政區序號及區名；正式區序代碼規格未知,官方頁列欄名；檔內觀測，不自行當里代碼,事故情境／事故總量（候選）,0,12
6,肇事地點,str,事故地點文字；不是當事人居住地址,PDF p.1 發生地點；發布 CSV 轉錄細節未知,事故情境／事故總量（候選）,0,15144
7,死亡人數,float64,事故死傷人數中的 24 小時內死亡人數；不含個人受傷程度代碼,PDF p.1 死傷人數；與官方頁 24 小時定義對照,事故情境／事故總量（候選）,19,3
8,2-30日死亡人數,float64,欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認,CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式,事故情境／事故總量（候選）,19,2
9,受傷人數,float64,事故受傷人數；與單一當事人的受傷程度不同,PDF p.1 死傷人數,事故情境／事故總量（候選）,2,12


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
16,道路型態,float64,事故道路型態代碼,PDF p.1；例 03 三岔、04 四岔、14 直路,事故情境／事故總量（候選）,19,14
17,事故位置,float64,事故發生位置類別,PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區,事故情境／事故總量（候選）,19,22
18,路面狀況1,float64,路面鋪裝代碼,PDF p.1 路面狀況 (1),事故情境／事故總量（候選）,19,5
19,路面狀況2,float64,路面狀態代碼,PDF p.1 路面狀況 (2)；5 乾燥,事故情境／事故總量（候選）,19,5
20,路面狀況3,float64,路面缺陷代碼,PDF p.1 路面狀況 (3)；4 無缺陷,事故情境／事故總量（候選）,19,4
21,道路障礙1,float64,障礙物代碼,PDF p.1 道路障礙 (1)；5 無障礙物,事故情境／事故總量（候選）,19,6
22,道路障礙2,float64,視距代碼；新版及表單內舊碼不可混用,PDF p.1 道路障礙 (2) 同時出現版本相關項目；13 良好,事故情境／事故總量（候選）,19,6
23,號誌1,float64,號誌種類代碼,PDF p.1 號誌 (1)；4 無號誌,事故情境／事故總量（候選）,19,4
24,號誌2,float64,號誌動作代碼,PDF p.1 號誌 (2)；1 正常、4 無號誌,事故情境／事故總量（候選）,19,4
25,車道劃分-分向,float64,分向設施代碼,PDF p.1 分向設施；10 無分向設施,事故情境／事故總量（候選）,19,10


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
32,受傷程度,float64,單一當事者傷亡程度，不是事故傷亡總數，也無法直接辨識重傷,PDF p.3；1 24小時內死亡、2 受傷、3 未受傷、4 不明、5 2–30日內死亡,當事人（候選）,1367,5
33,主要傷處,float64,當事者主要受傷部位代碼,PDF p.3；09 多數傷、10 無、11 不明,當事人（候選）,1367,11
34,保護裝置,float64,當事者保護裝備使用類別,PDF p.3 保護裝備；3 不明、4 無需使用等，版本映射仍需驗證,當事人（候選）,1373,6
35,行動電話,float64,行動電話、電腦或類似裝置使用情形,PDF p.3；1 未使用、4 不明、5 非駕駛人,當事人（候選）,1370,5
36,車輛用途,float64,車輛用途／非駕駛人類別,PDF p.3；7 其他、8 非駕駛人及乘客,當事人（候選）,1364,7
37,當事者行動狀態,float64,車或人的事故當時行動狀態,PDF p.3；9 向前直行、16 步行、21 不明,當事人（候選）,1366,21
38,駕駛資格情形,float64,駕照狀態代碼,PDF p.3；1 正常駕照、7 不明、8 無需駕照之人,當事人（候選）,1362,7
39,駕駛執照種類,float64,駕照種類代碼,PDF p.3；18 無駕照、19 不明、20 無需駕照之人,當事人（候選）,1382,20
40,飲酒情形,float64,酒精觀察／檢測分類，不是連續酒測值,PDF p.3；1 觀察未飲酒、2 無反應、9 無法檢測、10 非駕駛人、11 不明,當事人（候選）,1084,11
41,車輛撞擊部位1,float64,最初撞擊部位候選對應,PDF p.3 車輛撞擊部位 最初／其他；CSV 對應待發布規格確認,當事人（候選）,1373,17


,欄位,讀入型態,定義或可觀察內容,定義依據及限制,層級,缺失數,非缺失相異值
48,座標-Y,float64,原始點位 Y 數值；範圍近似緯度但座標基準未確認,官方頁僅稱 X、Y；不據範圍自行指定 EPSG,事故情境／事故總量（候選）,2,22042


**表 5：2024 原始描述統計（代碼均值不作實質解讀）**

... (7 outputs left out)

**表 5：2025 原始描述統計（代碼均值不作實質解讀）**

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
發生年度,22762.0,NaN,NaN,NaN,114.0,0.0,114.0,114.0,114.0,114.0,114.0
發生月,22762.0,NaN,NaN,NaN,6.609437,3.471728,1.0,4.0,7.0,10.0,12.0
發生日,22762.0,NaN,NaN,NaN,15.678543,8.615397,1.0,8.0,16.0,23.0,31.0
發生時-Hours,22762.0,NaN,NaN,NaN,13.333275,5.153732,0.0,9.0,13.0,18.0,23.0
發生分,22762.0,NaN,NaN,NaN,28.859151,17.268878,0.0,14.0,30.0,44.0,59.0
區序,22762,12,03中山區,3015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
肇事地點,22762,15266,大同區鄭州路與西寧北路口,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
死亡人數,22755.0,NaN,NaN,NaN,0.002549,0.051287,0.0,0.0,0.0,0.0,2.0
2-30日死亡人數,22752.0,NaN,NaN,NaN,0.000132,0.011482,0.0,0.0,0.0,0.0,1.0
受傷人數,22761.0,NaN,NaN,NaN,1.326875,0.597007,0.0,1.0,1.0,2.0,21.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
路面狀況3,22750.0,NaN,NaN,NaN,3.998593,0.081729,1.0,4.0,4.0,4.0,5.0
道路障礙1,22750.0,NaN,NaN,NaN,4.964264,0.320443,1.0,5.0,5.0,5.0,6.0
道路障礙2,22750.0,NaN,NaN,NaN,12.964967,0.326768,8.0,13.0,13.0,13.0,13.0
號誌1,22750.0,NaN,NaN,NaN,2.908,1.265954,1.0,2.0,4.0,4.0,4.0
號誌2,22750.0,NaN,NaN,NaN,2.650462,1.490581,1.0,1.0,4.0,4.0,4.0
車道劃分-分向,22750.0,NaN,NaN,NaN,5.371604,3.645113,1.0,1.0,5.0,9.0,10.0
車道劃分-分道1,22750.0,NaN,NaN,NaN,4.192571,0.75906,1.0,4.0,4.0,5.0,5.0
車道劃分-分道2,22750.0,NaN,NaN,NaN,4.813275,0.647674,1.0,5.0,5.0,5.0,5.0
車道劃分-分道3,22750.0,NaN,NaN,NaN,1.824967,0.380004,1.0,2.0,2.0,2.0,2.0
事故類型及型態,22750.0,NaN,NaN,NaN,14.486549,3.937048,1.0,13.0,16.0,17.0,38.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
飲酒情形,22656.0,NaN,NaN,NaN,2.077198,2.537888,1.0,1.0,1.0,2.0,11.0
車輛撞擊部位1,22557.0,NaN,NaN,NaN,9.337412,4.892926,1.0,5.0,11.0,13.0,17.0
肇因碼-個別,22739.0,NaN,NaN,NaN,57.757861,38.618416,1.0,16.0,70.0,100.0,131.0
個人肇逃否,22740.0,NaN,NaN,NaN,1.038874,0.193299,1.0,1.0,1.0,1.0,2.0
職業,22551.0,NaN,NaN,NaN,21.346104,0.952198,14.0,21.0,21.0,21.0,29.0
座標-X,22760.0,NaN,NaN,NaN,121.540491,0.030742,121.46118,121.516802,121.536655,121.562316,121.652018
座標-Y,22760.0,NaN,NaN,NaN,25.05433,0.032454,24.96593,25.034563,25.049958,25.068917,25.183295


#### 觀察單位、候選識別鍵及跨年比較
候選鍵為年月日時分＋肇事地點＋X／Y；分組時保留缺座標群組（dropna=False）。
群內一致性及第一當事人唯一性只能支持操作定義，不能證明現實世界事件沒有同時同地碰撞。
不依候選鍵去重、不輸出事故表；2024 傷亡欄在當事人間重複，不能按原始列直接加總。


In [6]:
KEY = ['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
EVENT = ['區序','死亡人數','2-30日死亡人數','受傷人數','天候','道路類別','速限-速度限制','道路型態','事故位置','事故類型及型態']
diagnostics=[]
first_frames={}
for year,frame in frames.items():
    first=frame.loc[frame['當事人序號'].eq(1)]
    first_frames[year]=first
    time=pd.to_datetime({'year':frame['發生年度']+1911,'month':frame['發生月'],
                         'day':frame['發生日'],'hour':frame['發生時-Hours'],
                         'minute':frame['發生分']},errors='coerce')
    grouped=frame.groupby(KEY,dropna=False,sort=False)
    first_counts=grouped['當事人序號'].agg(lambda s:int(s.eq(1).sum()))
    conflicts=grouped[EVENT].nunique(dropna=False).gt(1).sum().rename('候選群組內不一致數').to_frame()
    show_table(f'表 6：{year} 當事人序號分布；分母=原始列數 {len(frame):,}',
               frame['當事人序號'].value_counts(dropna=False).sort_index().rename('列數').to_frame())
    show_table(f'表 7：{year} 候選時地群組內事故欄位一致性（含缺失為類別）',conflicts)
    months=pd.concat([frame['發生月'].value_counts().sort_index().rename('原始列'),
                      first['發生月'].value_counts().sort_index().rename('第一當事人候選紀錄')],axis=1)
    show_table(f'表 8：{year} 月份涵蓋，非事故率',months)
    diagnostics.append({'年度':year,'原始列':len(frame),'欄數':len(frame.columns),
        '完整列重複':int(frame.duplicated().sum()),'第一當事人候選列':len(first),
        '候選時地群組':grouped.ngroups,'群組第一當事人非恰一筆':int(first_counts.ne(1).sum()),
        '第一當事人候選鍵重複':int(first.duplicated(KEY).sum()),
        '日期時間無法解析':int(time.isna().sum()),'最早時間':str(time.min()),'最晚時間':str(time.max()),
        '候選列缺至少一座標':int(first[['座標-X','座標-Y']].isna().any(axis=1).sum())})
show_table('表 9：事故原始單位與候選事件診斷',pd.DataFrame(diagnostics))
comparison=[]
for year,expected_rows,expected_columns,expected_first in [(2024,51810,49,22368),(2025,22762,47,22762)]:
    frame=frames[year]
    for metric,expected,observed in [('原始列',expected_rows,len(frame)),('欄數',expected_columns,len(frame.columns)),
                                    ('第一當事人候選列',expected_first,len(first_frames[year]))]:
        comparison.append({'年度':year,'項目':metric,'Unit 0.2 紀錄':expected,'本次觀測':observed,'一致':expected==observed})
candidate_total=sum(len(f) for f in first_frames.values())
missing_total=sum(int(f[['座標-X','座標-Y']].isna().any(axis=1).sum()) for f in first_frames.values())
comparison += [{'年度':'兩年','項目':'候選紀錄合計','Unit 0.2 紀錄':45130,'本次觀測':candidate_total,'一致':candidate_total==45130},
               {'年度':'兩年','項目':'候選紀錄缺座標','Unit 0.2 紀錄':3,'本次觀測':missing_total,'一致':missing_total==3}]
show_table('表 10：與 Unit 0.2 對照；不以先前數值強制篩選或補值',pd.DataFrame(comparison))
print({'candidate_total':candidate_total,'candidate_missing_coordinates':missing_total,'not_final_GIS_matched_sample':True})


**表 6：2024 當事人序號分布；分母=原始列數 51,810**

,列數
當事人序號,
1,22368
2,22367
3,5239
4,1104
5,350
6,139
7,73
8,46
9,32


,列數
當事人序號,
26,1
27,1


**表 7：2024 候選時地群組內事故欄位一致性（含缺失為類別）**

,候選群組內不一致數
區序,0
死亡人數,0
2-30日死亡人數,0
受傷人數,0
天候,0
道路類別,0
速限-速度限制,0
道路型態,0
事故位置,0
事故類型及型態,0


**表 8：2024 月份涵蓋，非事故率**

... (8 outputs left out)

,年度,原始列,欄數,完整列重複,第一當事人候選列,候選時地群組,群組第一當事人非恰一筆,第一當事人候選鍵重複,日期時間無法解析,最早時間,最晚時間,候選列缺至少一座標
0,2024,51810,49,0,22368,22368,0,0,0,2024-01-01 00:22:00,2024-12-31 23:46:00,1
1,2025,22762,47,0,22762,22762,0,0,0,2025-01-01 00:18:00,2025-12-31 23:22:00,2


**表 10：與 Unit 0.2 對照；不以先前數值強制篩選或補值**

,年度,項目,Unit 0.2 紀錄,本次觀測,一致
0,2024,原始列,51810,51810,True
1,2024,欄數,49,49,True
2,2024,第一當事人候選列,22368,22368,True
3,2025,原始列,22762,22762,True
4,2025,欄數,47,47,True
5,2025,第一當事人候選列,22762,22762,True
6,兩年,候選紀錄合計,45130,45130,True
7,兩年,候選紀錄缺座標,3,3,True


{'candidate_total': 45130, 'candidate_missing_coordinates': 3, 'not_final_GIS_matched_sample': True}


#### 缺失、特殊值及跨年候選照明映射


In [7]:
for year,frame in frames.items():
    first=first_frames[year]
    casualty=['死亡人數','2-30日死亡人數','受傷人數','受傷程度']
    q=pd.DataFrame({'原始非缺失 N':frame[casualty].notna().sum(),'原始缺失':frame[casualty].isna().sum(),
                    '候選第一當事人缺失':first[casualty].isna().sum()})
    show_table(f'表 11：{year} 傷亡欄位有效數；不補 0',q)
    show_table(f'表 12：{year} 座標原始範圍；未指定 CRS',frame[['座標-X','座標-Y']].agg(['count','min','max']).T)
    show_table(f'表 13：{year} 速限原始值分布；未修正 0／502',
               frame['速限-速度限制'].value_counts(dropna=False).sort_index().rename('原始列數').to_frame())
    age=frame['年齡']
    show_table(f'表 14：{year} 年齡診斷；特殊值含義待確認',pd.DataFrame([{
        '原始最小':age.min(),'原始最大':age.max(),'負值':int(age.lt(0).sum()),
        '零':int(age.eq(0).sum()),'大於100':int(age.gt(100).sum()),'缺失':int(age.isna().sum())}]))
    lighting='道路照明設備' if year==2024 else '光線'
    show_table(f'表 15：{year} {lighting} 實際代碼分布；未跨年重編碼',
               frame[lighting].value_counts(dropna=False).sort_index().rename('原始列數').to_frame())
    for column in ['性別','受傷程度','主要傷處','保護裝置','飲酒情形','事故類型及型態']:
        show_table(f'表 16：{year} {column}（缺失與特殊類別分開）',
                   frame[column].value_counts(dropna=False).sort_index().rename('列數').to_frame())


**表 11：2024 傷亡欄位有效數；不補 0**

,原始非缺失 N,原始缺失,候選第一當事人缺失
死亡人數,51791,19,9
2-30日死亡人數,51791,19,9
受傷人數,51808,2,1
受傷程度,50443,1367,227


**表 12：2024 座標原始範圍；未指定 CRS**

,count,min,max
座標-X,51808.0,121.461189,121.643370
座標-Y,51808.0,24.966712,25.188246


**表 13：2024 速限原始值分布；未修正 0／502**

,原始列數
速限-速度限制,
0.0,710
10.0,2
15.0,5
20.0,30
25.0,45
30.0,10956
40.0,5184
50.0,34185
59.0,2


... (36 outputs left out)

,列數
飲酒情形,
1.0,14921
2.0,5547
3.0,18
4.0,17
5.0,11
6.0,23
7.0,35
8.0,41
9.0,1050


**表 16：2025 事故類型及型態（缺失與特殊類別分開）**

,列數
事故類型及型態,
1.0,137
2.0,198
3.0,889
4.0,4
5.0,15
6.0,39
7.0,14
8.0,25
9.0,725


,列數
事故類型及型態,
29.0,143
35.0,9
36.0,7
37.0,5
38.0,17
NaN,12


#### Reference Data — 事故對照表 PDF
本機 PDF 與兩年度 CSV 共用此文件，仍在 PROJECT.md 逐檔登記，無須另建空 Notebook。
PDF 為表單／代碼說明，不是事故微觀資料；頁數與文字長度不當作觀察值數或資料欄數。
PDF 的原始網路下載時間及實際下載來源未取得；官方事故頁備註提供 112.7.1 後適用表單連結，尚未下載比對。

官方事故頁列出的[112.7.1 後適用表單](https://www-ws.gov.taipei/001/Upload/467/relfile/32284/5040008/7ddb52e9-ebae-4834-b41a-5f41d5074279.pdf)僅作來源線索；本次未下載或認證其與本機 PDF 的位元組一致性。


In [8]:
import pymupdf
with pymupdf.open(project_file('data/raw/事故對照表.pdf')) as document:
    pages=[page.get_text() for page in document]
    print({'PDF_pages':len(pages),'page_text_lengths':[len(t) for t in pages],
           'printed_effective_version':'112.07.01 後適用' if '112.07.01' in pages[0] else '待確認',
           'remote_byte_identity':'未重新下載比對；不宣稱本機 PDF 與官方當前下載位元組相同'})
    assert len(document)==4
    assert '112.07.01' in pages[0]
show_table('表 17：本機 PDF 主要代碼與候選對應（定義須保留適用版本限制）',pd.DataFrame([
 {'欄位':'道路照明設備／光線','PDF頁':'1','代碼':'1 日間自然光；2 晨暮；3／4 夜間隧道等有／無照明；5 有照明且開啟；6 未開啟或故障；7 無照明','限制':'兩年只有 5–7 非缺失值，不因此認證兩欄完全同義或可辨識日夜'},
 {'欄位':'事故位置','PDF頁':'1','代碼':'1 路口內、2 附近、3 待轉、4 停等','限制':'候選 intersection 包含 1–4，不能稱嚴格路口內'},
 {'欄位':'事故類型及型態','PDF頁':'1–2','代碼':'1–9 人與車；13 追撞；16 側撞','限制':'側撞含轉向情境，與 11／12 擦撞不同'},
 {'欄位':'受傷程度','PDF頁':'3','代碼':'1 24小時內死亡；2 受傷；3 未傷；4 不明；5 2–30日死亡','限制':'屬當事人，不代替事故總人數，無重傷識別'},
 {'欄位':'肇因碼','PDF頁':'3–4','代碼':'個別／主要初步分析研判索引','限制':'不作因果或最終責任認定'}]))


{'PDF_pages': 4, 'page_text_lengths': [2411, 500, 2801, 1941], 'printed_effective_version': '112.07.01 後適用', 'remote_byte_identity': '未重新下載比對；不宣稱本機 PDF 與官方當前下載位元組相同'}


**表 17：本機 PDF 主要代碼與候選對應（定義須保留適用版本限制）**

,欄位,PDF頁,代碼,限制
0,道路照明設備／光線,1,1 日間自然光；2 晨暮；3／4 夜間隧道等有／無照明；5 有照明且開啟；6 未開啟或故障；7 無照明,兩年只有 5–7 非缺失值，不因此認證兩欄完全同義或可辨識日夜
1,事故位置,1,1 路口內、2 附近、3 待轉、4 停等,候選 intersection 包含 1–4，不能稱嚴格路口內
2,事故類型及型態,1–2,1–9 人與車；13 追撞；16 側撞,側撞含轉向情境，與 11／12 擦撞不同
3,受傷程度,3,1 24小時內死亡；2 受傷；3 未傷；4 不明；5 2–30日死亡,屬當事人，不代替事故總人數，無重傷識別
4,肇因碼,3–4,個別／主要初步分析研判索引,不作因果或最終責任認定


#### 診斷結論與下一步
本次重算與 Unit 0.2 的六項主要筆數一致；沒有為符合紀錄而篩選或修補原始值。
兩年共同 46 欄；2024 額外有道路照明設備、撞擊部位2、主要肇因，2025 另有光線。
第一當事人不是「全部當事人」，其年齡、性別、車種不能代表全部涉入者；也不假設居住在事故發生里。
年齡負值／零／高齡、速限 0／502、傷亡及分類缺失、未認證座標 CRS 皆保留至 Chapter 1 審查。
不得把只有死傷案件的資料當作所有道路事故、交通流量或暴露校正事故率。


## Cross-Dataset Documentation — 僅規劃、未正式整合

| 原始資料 | 原始觀察單位／時間／空間粒度 | 預計連接與可支持變數 | 整合前條件 |
| --- | --- | --- | --- |
| 2024、2025 事故 | 2024 為當事人紀錄；2025 僅第一當事人；事故時分與點位 | 第一當事人候選事件；驗證 CRS 後以點在行政里面歸里；A 事故組成、C 負擔 | 官方事件口徑、照明欄位跨年映射、缺失／異常及座標基準；邊界點／未匹配規則另審查。 |
| 人口 | 年×月×區域×性別，含市／區／里 | 里層級「計」；12 月平均存量、人口結構 B1 | 代碼 crosswalk、糖廍里例外、月資料基準日與里界適用性；不視為交通暴露人口。 |
| 行政里界 | 456 個里面單元、單一未認證年代快照 | CPTVID 候選空間 ID；人口名稱及代碼驗證後連接，並作事故歸里 | 適用年度、幾何重疊／縫隙、原 AREA 定義與重算口徑；不因里數相同認證兩年幾何一致。 |
| 都市計畫 | 細計面與點、未認證年代快照、未知 CRS | 確認後以面圖層分類與疊合支援 B2 分區比例；點圖層用途待確認 | 原包、CRS、版本、143 筆無效幾何、重複及部分重疊、非一般分區、正式分類表。 |

| 規劃資料集 | 來源及變數 | 狀態與解釋限制 |
| --- | --- | --- |
| Dataset A | 事故；早／晚尖峰、深夜、週末、路口、追撞、側撞、人車、多人受傷組成 | 尚未建立；保留分子／有效分母／缺失／涵蓋率，不用總人口作事故組成分母。 |
| Dataset B | 人口 B1＋通過驗證的都市計畫 B2；其他道路環境 B3 尚未取得或選用 | 尚未建立；不納入事故結果，不以事故中的道路情境代表全里環境；分區不是實際土地使用。 |
| Dataset C | 事故數、傷亡、每件受傷數、事故空間密度、低樣本標記 | 尚未建立；24 小時與 2–30 日死亡分列，缺值不補成 0；每平方公里事故數不是暴露校正事故率。 |

village-year 為後續基礎資料結構；兩年比例合併須合計分子／有效分母再重算，
不能直接平均年度比例。零事故里及低樣本門檻仍待審查。本文件未建立 crosswalk、
年度人口表或 A／B／C，亦未執行事故歸里、土地使用疊合、分群或統計推論。

其他原始資料說明：
[事故](taipei_2024_2025_crashes_raw_documentation.ipynb)、[人口](taipei_population_age_raw_documentation.ipynb)、[里界](taipei_village_boundaries_raw_documentation.ipynb)、[都市計畫](taipei_urban_planning_raw_documentation.ipynb).


In [9]:
finish_check()


{'input_hashes_unchanged': True, 'input_files_checked': 3, 'interpreter_matches_project_venv': True, 'python_version': '3.12.2', 'scope': 'read-only raw documentation; no processed dataset or model output'}
